In [4]:
import sqlite3
import os
import random

In [10]:
# Various SQL Create Commands
SQL_User = '''CREATE TABLE IF NOT EXISTS User (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  username TEXT NOT NULL UNIQUE,
  password TEXT NOT NULL,
  created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP
)'''

SQL_Species = '''CREATE TABLE IF NOT EXISTS Species (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  name TEXT NOT NULL,
  emoji TEXT NOT NULL,
  hunger_rate INTEGER NOT NULL CHECK (hunger_rate >= 0),
  happiness_rate INTEGER NOT NULL CHECK (happiness_rate >= 0),
  energy_rate INTEGER NOT NULL CHECK (energy_rate >= 0),
  xp_multiplier REAL NOT NULL CHECK (xp_multiplier > 0)
)'''

SQL_Pets = '''CREATE TABLE IF NOT EXISTS Pets (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  owner_id INTEGER NOT NULL,
  species_id INTEGER NOT NULL,
  name TEXT NOT NULL,
  hunger INTEGER NOT NULL DEFAULT 100 CHECK (hunger BETWEEN 0 AND 100),
  happiness INTEGER NOT NULL DEFAULT 100 CHECK (happiness BETWEEN 0 AND 100),
  energy INTEGER NOT NULL DEFAULT 100 CHECK (energy BETWEEN 0 AND 100),
  xp INTEGER NOT NULL DEFAULT 0 CHECK (xp >= 0),
  level INTEGER NOT NULL DEFAULT 1 CHECK (level >= 1),
  last_updated DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
  born_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
  FOREIGN KEY (owner_id) REFERENCES User(id),
  FOREIGN KEY (species_id) REFERENCES Species(id)
)'''

SQL_Pets_Update_Trigger = '''CREATE TRIGGER IF NOT EXISTS update_pets_last_updated
AFTER UPDATE OF owner_id, species_id, name, hunger, happiness, energy, xp, level ON Pets
FOR EACH ROW
BEGIN
  UPDATE Pets
  SET last_updated = CURRENT_TIMESTAMP
  WHERE id = NEW.id AND last_updated IS NOT CURRENT_TIMESTAMP;
END;'''

SQL_Activity = '''CREATE TABLE IF NOT EXISTS Activity (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  pet_id INTEGER NOT NULL,
  action TEXT NOT NULL,
  created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
  FOREIGN KEY (pet_id) REFERENCES Pets(id)
)'''

In [6]:
# set up database connection
conn = sqlite3.connect('../database/pet.db')
conn.execute("PRAGMA foreign_keys = 1")
c = conn.cursor()

In [16]:
# reset tables
def reset_tables():
    c.execute('DROP TABLE IF EXISTS Activity')
    c.execute('DROP TABLE IF EXISTS Pets')
    c.execute('DROP TABLE IF EXISTS Species')
    c.execute('DROP TABLE IF EXISTS User')
    conn.commit()

def show_tables():
    c.execute('''
    SELECT name 
    FROM sqlite_master 
    WHERE type='table' 
    ORDER BY name;
    ''')
    print(c.fetchall())


In [23]:
# check tables
reset_tables()
for table in [SQL_User, SQL_Species, SQL_Pets, SQL_Activity]:
    c.execute(table)
c.execute(SQL_Pets_Update_Trigger)
conn.commit()

In [39]:
# add one record to each table
def add_user(username, password):
    c.execute(
        'INSERT INTO User (username, password) VALUES (?, ?)',
        (username, password)
    )
    conn.commit()

def add_species(name, emoji, hunger_rate, happiness_rate, energy_rate, xp_multiplier):
    c.execute(
        'INSERT INTO Species (name, emoji, hunger_rate, happiness_rate, energy_rate, xp_multiplier) VALUES (?, ?, ?, ?, ?, ?)',
        (name, emoji, hunger_rate, happiness_rate, energy_rate, xp_multiplier)
    )
    conn.commit()

def add_pet(owner_id, species_id, name, hunger=100, happiness=100, energy=100, xp=0, level=1):
    c.execute(
        'INSERT INTO Pets (owner_id, species_id, name, hunger, happiness, energy, xp, level) VALUES (?, ?, ?, ?, ?, ?, ?, ?)',
        (owner_id, species_id, name, hunger, happiness, energy, xp, level)
    )
    conn.commit()

def add_activity(pet_id, action):
    c.execute(
        'INSERT INTO Activity (pet_id, action) VALUES (?, ?)',
        (pet_id, action)
    )
    conn.commit()
    
# get all records from each table
def get_users():
    c.execute('SELECT * FROM User')
    return c.fetchall()

def get_species():
    c.execute('SELECT * FROM Species')
    return c.fetchall()

def get_pets():
    c.execute('SELECT * FROM Pets')
    return c.fetchall()

def get_activities():
    c.execute('SELECT * FROM Activity')
    return c.fetchall()

def adopt_candidates():
    c.execute('SELECT COUNT(*) FROM Species')
    species_count = c.fetchone()[0]
    return [random.randint(1, species_count) for x in range(4)]

In [17]:
show_tables()

In [38]:
# add sample species records
add_species('British Shorthair Cat', '🐱', 2, 1, 1, 1.1)
add_species('Golden Retriever Dog', '🐶', 3, 2, 2, 1.2)
add_species('Netherland Dwarf Rabbit', '🐰', 2, 2, 3, 1.0)
add_species('Syrian Hamster', '🐹', 1, 1, 2, 0.9)
add_species('Sun Conure Parrot', '🦜', 2, 3, 2, 1.1)
add_species('Red-Eared Slider Turtle', '🐢', 1, 1, 1, 0.8)

IntegrityError: UNIQUE constraint failed: User.username

In [40]:
get_users()

[(1, 'jp', 'ilovehangrypets', '2026-11-18 06:02:36')]